In [ ]:
import os
import numpy as np
import rasterio
from rasterio.enums import Resampling

drive_dir = contentdriveMyDriveColab Notebooks2stuff
patches_x_dir = os.path.join(drive_dir, patchesX_features)
patches_y_dir = os.path.join(drive_dir, patchesY_labels)

# 1. Paths to raw imagery files
early_b4_path = os.path.join(drive_dir, bsi_output.tif) # We can use the reference geometry from these
# Replace these strings with your actual paths to the RAW JP2TIF bands if named differently
early_b4_file = os.path.join(drive_dir, contentdriveMyDriveColab Notebooks2stuffsoil_b4.JP2)
early_b8a_file = os.path.join(drive_dir, contentdriveMyDriveColab Notebooks2stuffsoil_b8A.JP2)
mid_b4_file = os.path.join(drive_dir, contentdriveMyDriveColab Notebooks2stuff2green_b4.jp2)
mid_b8a_file = os.path.join(drive_dir, contentdriveMyDriveColab Notebooks2stuff2green_b8A.jp2)

# Helper function to upscale 20m bands to 10m on the fly
def read_and_resample(band_path, target_meta)
    with rasterio.open(band_path) as src
        return src.read(
            1,
            out_shape=(target_meta['height'], target_meta['width']),
            resampling=Resampling.bilinear
        ).astype(np.float32)

print(Loading raw seasonal bands into an architecture stack...)
with rasterio.open(os.path.join(drive_dir, final_maize_mask.tif)) as src
    label_mask = src.read(1)
    meta_10m = src.meta.copy()

# Read all 4 raw structural layers
e_b4 = read_and_resample(early_b4_file, meta_10m)
e_b8a = read_and_resample(early_b8a_file, meta_10m)
m_b4 = read_and_resample(mid_b4_file, meta_10m)
m_b8a = read_and_resample(mid_b8a_file, meta_10m)

# Stack them to form our brand new [4, Height, Width] input space
raw_feature_stack = np.stack([e_b4, e_b8a, m_b4, m_b8a], axis=0)

_, img_height, img_width = raw_feature_stack.shape
patch_size = 256
stride = 256
patch_count = 0

print(Generating clean 4-channel raw patches on Google Drive...)
for y in range(0, img_height - patch_size + 1, stride)
    for x in range(0, img_width - patch_size + 1, stride)

        patch_x = raw_feature_stack[, yy+patch_size, xx+patch_size]
        patch_y = label_mask[yy+patch_size, xx+patch_size]

        if np.max(patch_y) == 0 and np.all(patch_x == 0)
            continue

        # Clean values and normalize raw reflectance data slightly (scaling down by 10000 typical for Sentinel)
        patch_x = np.nan_to_num(patch_x, nan=0.0)  10000.0

        np.save(os.path.join(patches_x_dir, fpatch_{patch_count}_x.npy), patch_x)
        np.save(os.path.join(patches_y_dir, fpatch_{patch_count}_y.npy), patch_y)

        patch_count += 1

print(fSuccess! Rewrote {patch_count} new raw spectral patches to Drive.)





In [ ]:
import os
import numpy as np
import torch
import rasterio

# Define paths
drive_dir = "/content/drive/MyDrive/Colab Notebooks/2stuff/"
model_load_path = os.path.join(drive_dir, "unet_maize_best2.pth")
output_raster_path = os.path.join(drive_dir, "unet_maize_prediction_raw_bands.tif")

print("Initializing 4-channel inference setup...")

# 1. Access master metadata reference using your final mask file
with rasterio.open(os.path.join(drive_dir, "final_maize_mask.tif")) as src:
    meta_prediction = src.meta.copy()

# Re-import helper function to scale/upsample bands on the fly
def read_and_resample(band_path, target_meta):
    with rasterio.open(band_path) as src:
        return src.read(
            1,
            out_shape=(target_meta['height'], target_meta['width']),
            resampling=rasterio.enums.Resampling.bilinear
        ).astype(np.float32)

# Load the exact same 4 raw bands used in training
e_b4 = read_and_resample(os.path.join(drive_dir, "/content/drive/MyDrive/Colab Notebooks/2stuff/soil_b4.JP2"), meta_prediction)
e_b8a = read_and_resample(os.path.join(drive_dir, "/content/drive/MyDrive/Colab Notebooks/2stuff/soil_b8A.JP2"), meta_prediction)
m_b4 = read_and_resample(os.path.join(drive_dir, "/content/drive/MyDrive/Colab Notebooks/2stuff/2green_b4.jp2"), meta_prediction)
m_b8a = read_and_resample(os.path.join(drive_dir, "/content/drive/MyDrive/Colab Notebooks/2stuff/2green_b8A.jp2"), meta_prediction)

# Stack them into the 4-channel configuration: [4, Height, Width]
feature_stack = np.stack([e_b4, e_b8a, m_b4, m_b8a], axis=0)
_, img_height, img_width = feature_stack.shape

# 2. Re-initialize U-Net structure with 4 input channels and load best weights
model = UNet(in_channels=4, out_channels=1)
model.load_state_dict(torch.load(model_load_path, map_location=device))
model = model.to(device)
model.eval()

# Empty canvas array to compile incoming predicted patches
prediction_canvas = np.zeros((img_height, img_width), dtype=np.float32)
patch_size = 256

print("Running 4-channel deep learning prediction across full map canvas...")
with torch.no_grad():
    for y in range(0, img_height, patch_size):
        for x in range(0, img_width, patch_size):

            y_end = min(y + patch_size, img_height)
            x_end = min(x + patch_size, img_width)

            # Slice patch from the raw stack
            patch_x = feature_stack[:, y:y_end, x:x_end]

            # Handle border padding constraints
            pad_y = patch_size - patch_x.shape[1]
            pad_x = patch_size - patch_x.shape[2]
            if pad_y > 0 or pad_x > 0:
                patch_x = np.pad(patch_x, ((0,0), (0, pad_y), (0, pad_x)), mode='constant')

            # Clean numeric scales down by 10000 exactly like training
            patch_x = np.nan_to_num(patch_x, nan=0.0) / 10000.0

            # Convert to tensor and deploy to active GPU
            input_tensor = torch.from_numpy(patch_x).float().unsqueeze(0).to(device)

            # Execute inference pass
            logits = model(input_tensor)
            probabilities = torch.sigmoid(logits).squeeze().cpu().numpy()

            actual_y_size = y_end - y
            actual_x_size = x_end - x
            valid_patch = probabilities[0:actual_y_size, 0:actual_x_size]

            # Splice patch into the finalized projection layer
            prediction_canvas[y:y_end, x:x_end] = valid_patch

# 3. Create a clean binary mask from model probability outputs
final_binary_output = np.where(prediction_canvas > 0.50, 1, 0).astype(np.uint8)

# 4. Save file back to Drive with transparent background parameters
meta_prediction.update(
    driver='GTiff',
    dtype=rasterio.uint8,
    count=1,
    nodata=0
)

with rasterio.open(output_raster_path, "w", **meta_prediction) as dst:
    dst.write(final_binary_output, 1)

print(f"\nSuccess! New AI map layer generated and saved as 'unet_maize_prediction_raw_bands.tif'")